# 02 — Parquet to DataFrames

Loads all 10 parquet files into pandas DataFrames, joins them into three denormalised DataFrames, and pickles them for use in the EDA notebooks.

**What it does:**
- Loads all parquet tables into memory and verifies row counts
- Flattens tags and labels into per-album and per-artist dictionaries
- Builds `final_album_df` — one row per album with ratings, country, label, and tag data joined in
- Builds `final_artist_df` — one row per artist with ratings and tags joined in
- Builds `master_df` — a full denormalised join of album and artist data in a single DataFrame
- Pickles all three DataFrames to `data/pickles/` for fast loading in EDA notebooks

**Inputs:** All 10 `data/mb_*.parquet` files

**Outputs to `data/pickles/`:** `final_album_df.pkl`, `final_artist_df.pkl`, `master_df.pkl`

**Run after:** `01-postgres-to-parquet.ipynb` | **Run before:** `03-EDA-albums.ipynb`, `04-EDA-artists.ipynb`, `05-EDA-master.ipynb`

## Step 1 — Load all 10 parquet files into memory

All source data lives in 10 separate parquet files under `data/`, each corresponding to a table extracted from MusicBrainz in the previous notebook. This cell loads all of them into pandas DataFrames in one go and prints a row count for each so you can verify nothing was dropped during the parquet export.

The data splits into two groups:

- **Artist tables:** `artists` (core artist metadata), `artist_tags` (genre/style tags with counts), `artist_ratings` (community ratings), `artist_credit` (the credit entity that links artists to albums — MusicBrainz uses an indirection layer here rather than a direct artist → album foreign key).
- **Album tables:** `albums` (core release group metadata), `album_tags`, `album_ratings`, `album_country` (release area), `album_label` (label metadata including any label-level tags), `album_artists` (the many-to-many mapping of albums to artists via artist credit).

Nothing is joined yet — each table is a flat, normalised slice of the original database schema.

In [ ]:
import pandas as pd

# Load Artist related data
artists = pd.read_parquet('../data/mb_artist.parquet')
artist_tags = pd.read_parquet('../data/mb_artist_tag.parquet')
artist_ratings = pd.read_parquet('../data/mb_artist_ratings.parquet')
artist_credit = pd.read_parquet('../data/mb_artist_credit.parquet')

# Load Album related data
albums = pd.read_parquet('../data/mb_album.parquet')
album_tags = pd.read_parquet('../data/mb_album_tag.parquet')
album_ratings = pd.read_parquet('../data/mb_album_ratings.parquet')
album_country = pd.read_parquet('../data/mb_album_country.parquet')
album_label = pd.read_parquet('../data/mb_album_label.parquet')
album_artists = pd.read_parquet('../data/mb_album_artists.parquet')

# Verify the loads
dataframes = {
    "Artists": artists,
    "Artist Tags": artist_tags,
    "Artist Ratings": artist_ratings,
    "Artist Credit": artist_credit,
    "Albums": albums,
    "Album Tags": album_tags,
    "Album Ratings": album_ratings,
    "Album Country": album_country,
    "Album Label": album_label,
    "Album Artists": album_artists
}

for name, df in dataframes.items():
    print(f"✅ {name}: {df.shape[0]:,} rows loaded.")

## Step 2 — Flatten tags into per-entity dictionaries

Tag tables in MusicBrainz are in long format: one row per `(entity_id, tag_id, tag_count)` triplet. This cell collapses them into a more useful shape — a single dict per entity mapping `tag_id → tag_count` — so that each album or artist ends up with one row rather than dozens.

Three separate dictionaries are built:

- **`album_tag_dict`** — tags applied directly to an album. Grouped by `album_id`, each value is `{tag_id: tag_count}`.
- **`artist_tag_dict`** — the artist's own tags, remapped onto albums. The join path is `albums.artist_credit → artist_tags.artist_id`: because MusicBrainz links albums to artists via the `artist_credit` indirection, the album's `artist_credit` field is used as the join key against `artist_tags.artist_id`. The resulting dict is still keyed by `album_id` so it can be merged into `final_album_df` later.
- **`label_tag_dict`** — tags attached to the record label of each album. These come from `album_label`, which already carries `tag_id` and `tag_count` columns alongside `album_id`.

Keeping tag counts (not just tag presence) is intentional — the recommendation model uses them as feature weights rather than binary flags.

In [4]:
# Flatten album_tags into {tag_id: tag_count} per album
album_tag_dict = (
    album_tags
    .groupby('album_id')
    .apply(lambda x: dict(zip(x['tag_id'], x['tag_count'])), include_groups=False)
    .reset_index()
    .rename(columns={0: 'album_tags'})
)

# Map artist_tags to albums via artist_credit, then flatten
artist_tag_dict = (
    albums[['id', 'artist_credit']]
    .merge(artist_tags, left_on='artist_credit', right_on='artist_id', how='inner')
    .groupby('id')
    .apply(lambda x: dict(zip(x['tag_id'], x['tag_count'])), include_groups=False)
    .reset_index()
    .rename(columns={'id': 'album_id', 0: 'artist_tags'})
)

# Flatten label_tags into {tag_id: tag_count} per album
label_tag_dict = (
    album_label[['album_id', 'tag_id', 'tag_count']]
    .groupby('album_id')
    .apply(lambda x: dict(zip(x['tag_id'], x['tag_count'])), include_groups=False)
    .reset_index()
    .rename(columns={0: 'label_tags'})
)

print(f'✅ album_tag_dict: {album_tag_dict.shape[0]:,} albums')
print(f'✅ artist_tag_dict: {artist_tag_dict.shape[0]:,} albums')
print(f'✅ label_tag_dict: {label_tag_dict.shape[0]:,} albums')

## Step 3 — Build `final_album_df`

This cell assembles the main album-level DataFrame used throughout the project. It starts from the `albums` table and progressively left-joins enrichment data onto it. A left join is used throughout so that albums with no ratings, no country, no label, or no tags are retained — missing data is represented as `NaN` rather than rows being silently dropped.

The join sequence is:

1. **`album_ratings`** — on `album_id`. Adds `rating` and `rating_count` columns from the community rating aggregate.
2. **`album_country`** — on `album_id`. Adds the ISO country/area code for the release.
3. **`album_label`** — on `album_id`, but only three columns are taken (`album_id`, `label_id`, `label_type`) to avoid pulling in the raw tag columns that now live in `label_tag_dict`.
4. **`album_tag_dict`** — on `album_id`. Adds the `album_tags` dict column built in Step 2.
5. **`label_tag_dict`** — on `album_id`. Adds the `label_tags` dict column built in Step 2.

Note that `artist_tag_dict` is **not** joined here — artist tags are added later in `master_df` when artist and album data are fully combined. The `id` column from `albums` is renamed to `album_id` and `name` to `album_name` upfront to avoid ambiguity in subsequent joins.

In [ ]:
# Build final_album_df: one row per album with scalar features
final_album_df = albums.rename(columns={'id': 'album_id', 'name': 'album_name'})
final_album_df = pd.merge(final_album_df, album_ratings, on='album_id', how='left')
final_album_df = pd.merge(final_album_df, album_country, on='album_id', how='left')
final_album_df = pd.merge(final_album_df, album_label[['album_id', 'label_id', 'label_type']], on='album_id', how='left')
final_album_df = pd.merge(final_album_df, album_tag_dict, on='album_id', how='left')
final_album_df = pd.merge(final_album_df, label_tag_dict, on='album_id', how='left')


print(f'✅ final_album_df: {final_album_df.shape[0]:,} rows, {final_album_df.shape[1]} columns')

## Step 4 — Build `final_artist_df`

This cell builds the artist-level DataFrame in two sequential left joins, starting from the `artists` table.

1. **`artist_ratings`** — joined on `artists.id = artist_ratings.artist_id`. After the merge, the redundant `artist_id` column from the ratings table is dropped, leaving `id` as the single artist key.
2. **Artist tags** — the `artist_tags` long table is collapsed inline (same groupby-apply pattern as Step 2) and joined on `artists.id = artist_tags.artist_id`. Again the duplicate `artist_id` column is dropped afterwards.

The inline tag flattening here (rather than using the pre-built `artist_tag_dict` from Step 2) is because `artist_tag_dict` was keyed by `album_id` — it remapped artist tags onto albums. This step builds a separate, artist-keyed version for the standalone artist DataFrame.

The output has one row per artist and includes all core artist metadata (`name`, `type`, `gender`, `area`, `artist_year`, etc.) plus `rating`, `rating_count`, and an `artist_tags` dict column.

In [ ]:
# Build final_artist_df: one row per artist with scalar features and tags
final_artist_df = pd.merge(artists, artist_ratings, left_on='id', right_on='artist_id', how='left').drop(columns=['artist_id'])
final_artist_df = pd.merge(
    final_artist_df,
    artist_tags.groupby('artist_id').apply(lambda x: dict(zip(x['tag_id'], x['tag_count'])), include_groups=False).reset_index().rename(columns={0: 'artist_tags'}),
    left_on='id',
    right_on='artist_id',
    how='left'
).drop(columns=['artist_id'])

print(f'✅ final_artist_df: {final_artist_df.shape[0]:,} rows, {final_artist_df.shape[1]} columns')

## Step 5 — Build `master_df`

`master_df` is the fully denormalised, single-table representation of the dataset — one row per album with all artist attributes joined in alongside all album attributes. This is the primary input for the recommendation model.

The join is `final_album_df.artist_credit = final_artist_df.id`. The `artist_credit` column on albums is MusicBrainz's indirection key that maps a release group back to its credited artist entity, so it acts as the foreign key into `final_artist_df`.

Only a subset of artist columns is pulled across to avoid bloat:
- `artist_year`, `type`, `area`, `gender` — demographic/biographical features
- `rating` and `rating_count` — renamed to `artist_rating` / `artist_rating_count` to distinguish them from the album-level equivalents already present in `final_album_df`
- `artist_tags` — the artist's tag dict, which complements the album and label tag dicts already in the DataFrame

After the merge, the `id` column (from `final_artist_df`) is dropped since `artist_credit` already serves as the link key and having both would be redundant.

The printed column list at the end is useful for confirming the full feature set before pickling.

In [5]:
# Build master_df: join final_album_df with final_artist_df
master_df = pd.merge(
    final_album_df,
    final_artist_df[['id', 'artist_year', 'type', 'area', 'gender', 'rating', 'rating_count', 'artist_tags']].rename(columns={'rating': 'artist_rating', 'rating_count': 'artist_rating_count'}),
    left_on='artist_credit',
    right_on='id',
    how='left'
).drop(columns=['id'])

print(f'\n🚀 master_df: {master_df.shape[0]:,} rows, {master_df.shape[1]} columns')
print(master_df.columns.tolist())


🚀 Master DataFrame ready with 157,945,861 rows.


## Step 6 — Pickle all three DataFrames

Serialises `final_album_df`, `final_artist_df`, and `master_df` to `data/pickles/` using pandas' pickle format.

Pickle is chosen over parquet or CSV here because these DataFrames contain dict-typed columns (`album_tags`, `artist_tags`, `label_tags`). Parquet does not support arbitrary Python dicts as a column dtype, and CSV would stringify them. Pickle preserves the exact Python objects, making subsequent notebooks fast to load without any re-parsing or reconstruction.

The `os.makedirs(..., exist_ok=True)` call ensures the `pickles/` directory is created if it doesn't already exist, so this cell is safe to run from a clean checkout.

Downstream notebooks (`03-EDA-albums.ipynb`, `04-EDA-artists.ipynb`, `05-EDA-master.ipynb`) load from these pickle files rather than re-running all the joins above.

In [ ]:
import os

os.makedirs('../data/pickles', exist_ok=True)

final_album_df.to_pickle('../data/pickles/final_album_df.pkl')
final_artist_df.to_pickle('../data/pickles/final_artist_df.pkl')
master_df.to_pickle('../data/pickles/master_df.pkl')

print("✅ final_album_df, final_artist_df and master_df have been pickled!")

## Step 7 — Inspect DataFrame schemas

Prints `.info()` for each of the raw source tables and for `master_df`. This gives a quick sanity check on column names, dtypes, and non-null counts across the full pipeline.

The raw source tables are included (not just the final DataFrames) so you can cross-reference what came in from parquet versus what ended up in the joined outputs. Things to watch for:

- Columns that should be fully populated (e.g. `album_id` in every table) showing unexpected nulls, which would indicate a join key mismatch upstream.
- The `album_tags`, `artist_tags`, and `label_tags` columns in `master_df` will show dtype `object` — that is expected since they hold Python dicts.
- `rating` and `rating_count` will have nulls in both album and artist tables because not all entities have been rated by the MusicBrainz community.

Note that `artist_credit` is intentionally omitted from the loop — it is a bridge table used only during joins and does not contribute columns to the final DataFrames.

In [ ]:
for name, df in {
    "artists": artists,
    "artist_tags": artist_tags,
    "artist_ratings": artist_ratings,
    "albums": albums,
    "album_tags": album_tags,
    "album_ratings": album_ratings,
    "album_country": album_country,
    "album_label": album_label,
    "album_artists": album_artists,
    "master_df": master_df
}.items():
    print(f"\n{'='*40}\n{name}\n{'='*40}")
    df.info()